# Pipeline CRM — Reportes Gold (Glue / PySpark)

Genera **4 reportes** a partir de Silver (última partición de cada tabla) y los escribe en `Gold/Drive/`:

| Reporte | Carpeta Gold |
|---|---|
| `reporte_oportunidades` | `ReporteOportunidades/` |
| `reporte_historico_oportunidades` | `ReporteHistoricoOportunidades/` |
| `reporte_citas` | `ReporteCitas/` |
| `reporte_simulaciones` | `ReporteSimulaciones/` |

Es un port 1:1 de `processedCrmAtlas.py` + `PipelineCrm_enhanced.ipynb` (pandas). Mismas columnas, mismo orden, mismas reglas.

**Pendiente / omitido a propósito:** el catálogo de usuarios SCD2 todavía no existe. Las columnas de equipo se escriben en nulo
(el esquema no cambia). Para activarlas, solo hay que reemplazar la función `add_team(...)` de la sección 3.

**Reglas de portado (diferencias pandas vs Spark que ya están resueltas aquí):**
- Orden con nulos explícito (`*_nulls_last`) en todos los `sort_values` + `drop_duplicates`.
- Condiciones null-safe (`eq`, `isin_`): en pandas `NaN == x` es False y `~False` es True; en Spark sería null.
- `_ord` (orden físico del parquet) como desempate donde pandas conserva el orden original (sorts de varias columnas son estables).
- Los joins con llaves nulas no empatan (en los datos actuales no hay nulos que empaten; se verifica con asserts).
- Las columnas que en pandas pasaban por `_filtrar_fecha` quedan como **timestamp** (dtype datetime64 en pandas).

In [ ]:
%idle_timeout 720
%glue_version 5.1
%worker_type G.1X
%number_of_workers 5

In [ ]:
import boto3
from datetime import datetime
from pyspark.sql import functions as F, Window
from pyspark.context import SparkContext
from awsglue.context import GlueContext

sc = SparkContext.getOrCreate()
glueContext = GlueContext(sc)
spark = glueContext.spark_session

# Las fechas de Silver son hora local "ingenua" (texto). UTC evita saltos por horario de verano al parsear / formatear.
spark.conf.set('spark.sql.session.timeZone', 'UTC')
spark.conf.set('spark.sql.legacy.timeParserPolicy', 'CORRECTED')   # patrón que no calza -> null (como errors='coerce')
spark.conf.set('spark.sql.ansi.enabled', 'false')
spark.conf.set('spark.sql.parquet.outputTimestampType', 'TIMESTAMP_MICROS')
spark.conf.set('spark.sql.shuffle.partitions', '16')
print('Spark listo.')

## 0. Parámetros y reglas de negocio (copiadas de `crm_config.py`)

In [ ]:
BUCKET   = 'aws-s3-reporte-atenea-usorg-nprd'
SILVER   = f's3://{BUCKET}/data/Silver/Atlas/'
GOLD     = f's3://{BUCKET}/data/Gold/Drive/'      # <- cambia a 'Gold/Drive_test/' para una corrida de prueba
ESCRIBIR = True                                    # False = calcula y valida pero no escribe

FECHA_CORTE          = '2026-07-01'   # oportunidades, citas, casos e históricos (estricto: > corte)
FECHA_CORTE_PEDIDOS  = '2026-06-01'   # pedidos y solicitudes de crédito
FECHA_RELLENO_CITAS  = '2026-08-17'   # relleno de perfilamiento en citas sin opportunity_id

COLS_OPORTUNIDADES = ['opportunity_id',
 'opportunity_name',
 'opportunity_created_date',
 'opportunity_created_date_day',
 'fecha_asignacion',
 'opportunity_owner',
 'owner_id',
 'opportunity_owner_equipo',
 'nombre_comprador',
 'id_am_comprador',
 'last_modified_date',
 'opportunity_source',
 'opportunity_source_aux',
 'opportunity_stage',
 'flag_caso_tomado_perf_sc',
 'asesor_caso_tomado_perf_sc',
 'equipo_asesor_caso_tomado_perf_sc',
 'fecha_asignacion_perfilamiento_sc',
 'fecha_caso_tomado_sc',
 'case_owner_name_perf_sc',
 'case_owner_id_perf_sc',
 'case_owner_equipo_perf_sc',
 'case_id_perfilamiento_sc',
 'case_status_perfilamiento_sc',
 'perf_bc_score',
 'perf_intencion_pago',
 'perf_contactado',
 'perf_interesado',
 'perf_comentarios',
 'perf_canal_respuesta',
 'perf_fecha_primer_contacto',
 'perf_fecha_ultimo_contacto',
 'perf_contacto_otro_momento',
 'perf_fecha_sig_contacto',
 'flag_perfilamento_credito',
 'asesor_perfilamiento_credito',
 'asesor_perfilamiento_credito_id',
 'equipo_asesor_perfilamiento_credito',
 'fecha_asignacion_perfilamiento_credito',
 'case_id_perfilamiento_credito',
 'case_status_perfilamiento_credito',
 'cred_contactado',
 'cred_perfilamiento_credit',
 'cred_canal_respuesta_credito',
 'numero_pedidos',
 'numero_pedidos_abiertos',
 'fecha_primer_pedido',
 'fecha_ultimo_pedido',
 'n_simulaciones_credito',
 'n_simulaciones_con_folio',
 'n_simulaciones_preaceptadas',
 'n_simulaciones_contingencia',
 'simulation_name_op',
 'created_date_sim_op',
 'tipo_credito',
 'flag_cita_agendada_oportunidad',
 'flag_cita_sin_pedido_agendada',
 'flag_cita_sin_pedido_show',
 'fecha_primera_cita_visita_comp',
 'fecha_ultima_cita_visita_comp',
 'flag_cita_show_oportunidad',
 'stage_antes_de_cierre',
 'usuario_que_cerro',
 'equipo_que_cerro',
 'fecha_cierre',
 'motivo_cierre_oportunidad',
 'detalle_motivo_cierre_oportunidad',
 'primer_booker_comprador',
 'equipo_primer_booker_comprador']

COLS_CITAS = ['numero_cita',
 'created_time',
 'created_date',
 'status',
 'sf_order_id',
 'rol',
 'work_type_name',
 'id_am',
 'nombre',
 'telefono',
 'email',
 'espacio_cita',
 'sched_date',
 'sched_start_time',
 'sched_end_time',
 'opportunity_id',
 'opportunity_name',
 'opportunity_created_date',
 'opportunity_owner_id',
 'opportunity_owner',
 'opportunity_owner_equipo',
 'opportunity_source',
 'opportunity_source_aux',
 'opportunity_stage',
 'fecha_asignacion',
 'perf_intencion_pago',
 'perf_bc_score',
 'perf_sc_case_owner_id',
 'perf_sc_case_owner_name',
 'booker_name',
 'booker_equipo',
 'booker_equipo_operativo',
 'status_cancelacion',
 'motivo_cancelacion',
 'cancelado_por_id',
 'cancelado_por',
 'cancelado_por_equipo',
 'cancelado_por_equipo_operativo',
 'check_in',
 'check_out',
 'fecha_reagendamiento',
 'flag_dummy',
 'flag_duplicada',
 'flag_cita_agendada_comprador',
 'flag_cita_show_comprador',
 'flag_booker_origen_nulo',
 'flag_oppid_origen_nulo']

COLS_SIMULACIONES = ['opportunity_id',
 'opportunity_name',
 'flag_eda',
 'tipo_credito',
 'proveedor_credito',
 'folio',
 'tipo_conclusion_flujo_credito',
 'simulation_name',
 'status_solicitud',
 'asesor',
 'id_am_comprador',
 'created_date',
 'simulation_id',
 'mensualidades',
 'folio_cotizacion',
 'mensualidad',
 'tasa_interes',
 'credito_aprobado',
 'enganche',
 'seguro_danos',
 'seguro_vida',
 'comision_apertura',
 'fecha_oferta',
 'tipo_pago_seguro_danos',
 'tipo_pago_seguro_vida',
 'ingreso_solicitud',
 'fecha_ingreso_solicitud',
 'fecha_vigencia',
 'fecha_dictamen',
 'fecha_cierre_credito',
 'tipo_inicio_flujo_credito',
 'motivo_cierre_credito',
 'reactivacion_credito',
 'comentario',
 'documentacion',
 'ajustado']

COLS_RELLENAR = ['opportunity_id',
 'opportunity_name',
 'opportunity_created_date',
 'opportunity_owner_id',
 'opportunity_owner',
 'opportunity_owner_equipo',
 'opportunity_source',
 'opportunity_source_aux',
 'opportunity_stage',
 'fecha_asignacion',
 'perf_intencion_pago',
 'perf_bc_score']

ID_AM_DUMMIES = [1067687, 1067696, 1067711, 1067720, 1067648, 1067654, 1067669, 1067678]

STATUS_PEDIDOS_ABIERTOS = ['revision de auto',
 'auto en cita',
 'negociacion de precio',
 'acuerdo de compraventa',
 'formalizacion de compra',
 'cita para entrega']

WT_COMPRADOR = ['cita inicial visita comprador', 'cita final busca vehiculo']

WT_VENDEDOR = ['cita recogida vehiculo',
 'cita final entrega vehiculo',
 'revision mecanica inicial',
 'revision mecanica final',
 'cita inicial entrega vehiculo',
 'recogida vehiculo']

CRITERIOS_SHOW = ['completa', 'en progreso']

CRITERIOS_CANCELADA = ['canceled', 'canceledbuyer', 'canceledseller', 'noshowbuyer', 'noshowseller', 'no show - buyer']

CRITERIOS_AGEND = {'rol': ['desconocido', 'comprador'], 'wtn': ['desconocido', 'cita inicial visita comprador']}

CRM_EQUIPOS_OPERATIVOS = {'patriotismo': 'espacios fisicos',
 'satelite': 'espacios fisicos',
 'torre_bbva': 'espacios fisicos',
 'gran sur': 'espacios fisicos',
 'torre bbva': 'espacios fisicos',
 'cdmx': 'sales center',
 'mid': 'sales center',
 'celula credito': 'sales center',
 'integration user': 'integration user'}

## 1. Helpers

In [ ]:
ACC_FROM = 'ÁÀÂÄÃÅáàâäãåÉÈÊËéèêëÍÌÎÏíìîïÓÒÔÖÕóòôöõÚÙÛÜúùûüÑñÇç'
ACC_TO   = 'AAAAAAaaaaaaEEEEeeeeIIIIiiiiOOOOOoooooUUUUuuuuNnCc'

def _c(c):
    return F.col(c) if isinstance(c, str) else c

def ts(c):
    """Texto -> timestamp (coerce). Equivale a pd.to_datetime(errors='coerce') sobre los formatos de Silver."""
    c = _c(c)
    return F.coalesce(F.to_timestamp(c, 'yyyy-MM-dd HH:mm:ss'),
                      F.to_timestamp(c, 'yyyy-MM-dd HH:mm'),
                      F.to_timestamp(c, 'yyyy-MM-dd'))

def eq(c, v):
    """pandas .eq(): null -> False (nunca null)."""
    return F.coalesce(_c(c) == F.lit(v), F.lit(False))

def isin_(c, vals):
    """pandas .isin(): null -> False (nunca null)."""
    return F.coalesce(_c(c).isin(list(vals)), F.lit(False))

def contains_(c, txt):
    """pandas .str.contains(na=False)."""
    return F.coalesce(_c(c).contains(txt), F.lit(False))

def nulo(tipo='string'):
    return F.lit(None).cast(tipo)

def fmt_d(c):   return F.date_format(_c(c), 'yyyy-MM-dd')
def fmt_dt(c):  return F.date_format(_c(c), 'yyyy-MM-dd HH:mm:ss')

def unaccent(c):
    """unidecode(...) para letras latinas con acento."""
    return F.translate(_c(c), ACC_FROM, ACC_TO)

def first_by(df, keys, order_cols):
    """drop_duplicates(keep='first') tras sort_values: row_number() con orden explícito."""
    w = Window.partitionBy(*keys).orderBy(*order_cols)
    return df.withColumn('_rn', F.row_number().over(w)).filter(F.col('_rn') == 1).drop('_rn')

def latest_partition(base):
    s3 = boto3.client('s3')
    bucket, prefix = base.replace('s3://', '').split('/', 1)
    pfx = prefix.rstrip('/') + '/'
    for _ in range(3):   # year, month, day
        r = s3.list_objects_v2(Bucket=bucket, Prefix=pfx, Delimiter='/')
        parts = [(int(p['Prefix'].rstrip('/').split('=')[-1]), p['Prefix']) for p in r.get('CommonPrefixes', []) if '=' in p['Prefix']]
        if not parts:
            raise FileNotFoundError(f'Sin particiones en {pfx}')
        pfx = max(parts)[1]
    return f's3://{bucket}/{pfx}'

def leer(nombre):
    p = latest_partition(SILVER + nombre + '/')
    print(f'  {nombre}: {p}')
    return spark.read.parquet(p).withColumn('_ord', F.monotonically_increasing_id())

def filtrar(df, col, corte):
    """Equivale a _filtrar_fecha: la columna pasa a datetime y se conserva solo > corte (invalidas -> fuera)."""
    return (df.withColumn(col, ts(col))
              .filter(F.col(col) > F.lit(corte).cast('timestamp')))

## 2. Lectura de Silver (última partición) y filtros de fecha

In [ ]:
print('Leyendo Silver...')
O   = leer('ProcOportunidades')
P   = leer('ProcPedidos')
C   = leer('ProcCasosCrm')
Ci  = leer('ProcCitas')
S   = leer('ProcSolicitudesCredito')
HC  = leer('ProcCasosCrmHistorico')
HCi = leer('ProcHistoricoCitas')
HO  = leer('ProcHistoricoOportunidades')
CL  = leer('ProcClientes')

O   = filtrar(O,   'opportunity_created_date', FECHA_CORTE)
Ci  = filtrar(Ci,  'created_date',             FECHA_CORTE)
HO  = filtrar(HO,  'created_date',             FECHA_CORTE)
HCi = filtrar(HCi, 'created_date',             FECHA_CORTE)
HC  = filtrar(HC,  'created_date',             FECHA_CORTE)
C   = filtrar(C,   'case_created_date',        FECHA_CORTE)
P   = filtrar(P,   'fecha_de_creacion',        FECHA_CORTE_PEDIDOS)
S   = filtrar(S,   'created_date',             FECHA_CORTE_PEDIDOS)

for _n, _d in dict(O=O, Ci=Ci, HO=HO, HCi=HCi, HC=HC, C=C, P=P, S=S).items():
    _d.cache()
    print(f'{_n:>4}: {_d.count():,}')

### Diagnósticos previos (solo informan; nada falla aquí)

In [ ]:
# 1) llaves únicas / nulas que condicionan los joins
_p_null = P.filter(F.col('commerce_order_id').isNull()).count()
_p_dup  = P.filter(F.col('commerce_order_id').isNotNull()).groupBy('commerce_order_id').count().filter('count > 1').count()
print(f'pedidos filtrados: commerce_order_id nulos={_p_null}, repetidos={_p_dup}  (ambos deberían ser 0 para que citas no duplique filas)')

# 2) fecha_asignacion debe venir dd/MM/yyyy
_fa_mal = O.filter(F.col('fecha_asignacion').isNotNull() & F.to_date('fecha_asignacion', 'dd/MM/yyyy').isNull()).count()
print(f'oportunidades con fecha_asignacion que no calza dd/MM/yyyy: {_fa_mal}')

# 3) opps con más de un caso de perfilamiento contact center (fan-out que pandas resuelve por orden)
_multi_sc = (C.filter(eq('case_subject', 'perfilamiento contact center') & F.col('opportunity_id').isNotNull())
              .groupBy('opportunity_id').count().filter('count > 1').count())
_multi_cr = (C.filter(eq('case_subject', 'perfilamiento credito') & F.col('opportunity_id').isNotNull())
              .groupBy('opportunity_id').count().filter('count > 1').count())
print(f'opps con >1 caso perfilamiento SC: {_multi_sc} | con >1 caso perfilamiento crédito: {_multi_cr}')

## 3. Catálogo de equipos (SCD2) — OMITIDO por ahora

Reemplazar esta función cuando exista el SCD2. Firma equivalente a `add_team_from_scd2(df, catalogo, date_col, user_id_col, output_col)`.

In [ ]:
def add_team(df, date_col, user_id_col, output_col):
    return df.withColumn(output_col, nulo('string'))

## 4. `reporte_simulaciones`

In [ ]:
rep_sim = (S.withColumn('flag_eda', F.coalesce(F.col('opportunity_name').startswith('CREDITO EDA'), F.lit(False)).cast('int'))
             .orderBy('_ord'))
_faltan = [c for c in COLS_SIMULACIONES if c not in rep_sim.columns]
rep_sim = rep_sim.select([c for c in COLS_SIMULACIONES if c in rep_sim.columns])
print(f'reporte_simulaciones: {rep_sim.count():,} filas, {len(rep_sim.columns)} cols')
if _faltan:
    print(f'  [INFO] columnas de la config que no existen en Silver y tampoco salen en pandas: {_faltan}')

## 5. `reporte_historico_oportunidades`

In [ ]:
rep_hist = (HO.filter(eq('field', 'stagename'))
              .select('opportunity_id', 'old_value', 'new_value',
                      F.upper('created_by').alias('created_by'), 'created_date', '_ord')
              .join(O.select('opportunity_id', 'opportunity_name'), 'opportunity_id', 'left')
              .orderBy('opportunity_id', 'created_date', '_ord')
              .drop('_ord'))
_n_hist = rep_hist.count()
assert _n_hist == HO.filter(eq('field', 'stagename')).count(), 'historico: el join con oportunidades duplicó filas'
print(f'reporte_historico_oportunidades: {_n_hist:,} filas, {len(rep_hist.columns)} cols')

## 6. `reporte_oportunidades`
### 6.1 Casos de perfilamiento y asesor que tomó el caso

In [ ]:
C_sc = (C.filter(eq('case_subject', 'perfilamiento contact center'))
          .withColumnRenamed('case_id', 'case_id_perfilamiento_sc')
          .withColumnRenamed('case_status', 'case_status_perfilamiento_sc')
          .withColumnRenamed('_ord', '_ord_sc'))

C_cr = (C.filter(eq('case_subject', 'perfilamiento credito'))
          .withColumnRenamed('case_id', 'case_id_perfilamiento_credito')
          .withColumnRenamed('case_status', 'case_status_perfilamiento_credito')
          .withColumnRenamed('_ord', '_ord_cr'))

HC_sc = HC.join(C_sc.select(F.col('case_id_perfilamiento_sc').alias('case_id')).distinct(), 'case_id', 'left_semi')
HC_cr = HC.join(C_cr.select(F.col('case_id_perfilamiento_credito').alias('case_id')).distinct(), 'case_id', 'left_semi')

_toma = lambda d: d.filter(eq('field', 'status') & eq('old_value', 'abierto') & eq('new_value', 'in progress'))

# primer cambio abierto -> in progress (quién tomó el caso y cuándo)
asesor_sc = (first_by(_toma(HC_sc), ['case_id'], [F.col('created_date').asc(), F.col('_ord').asc()])
             .select(F.col('case_id').alias('case_id_perfilamiento_sc'),
                     F.col('created_by_id').alias('asesor_perfilamiento_sc_id'),
                     F.upper('created_by').alias('asesor_caso_tomado_perf_sc'),
                     fmt_dt('created_date').alias('fecha_caso_tomado_sc')))

asesor_cr = (first_by(_toma(HC_cr), ['case_id'], [F.col('created_date').asc(), F.col('_ord').asc()])
             .select(F.col('case_id').alias('case_id_perfilamiento_credito'),
                     F.upper('created_by').alias('asesor_perfilamiento_credito'),
                     F.col('created_by_id').alias('asesor_perfilamiento_credito_id'),
                     fmt_dt('created_date').alias('fecha_asignacion_perfilamiento_credito')))

# cambios de dueño "caso de servicio" -> alguien (SIN deduplicar: igual que pandas; el dedupe se hace por oportunidad al final)
fecha_asig_sc = (HC_sc.filter(eq('field', 'owner') & eq('old_value', 'caso de servicio') & ~eq('new_value', 'caso de servicio'))
                 .select(F.col('case_id').alias('case_id_perfilamiento_sc'),
                         fmt_d('created_date').alias('fecha_asignacion_perfilamiento_sc'),
                         F.col('_ord').alias('_ord_ev')))
print('listo')

### 6.2 Resúmenes por oportunidad: citas comprador, pedidos, solicitudes, cierre, primer booker

In [ ]:
WT_COMPRADOR_OPP = ['cita inicial visita comprador', '']

def _wt_comprador_opp():
    return isin_('work_type_name', WT_COMPRADOR_OPP) | F.col('work_type_name').isNull()

# citas de comprador con oportunidad
ci_comp = (Ci.filter(F.col('opportunity_id').isNotNull())
             .filter(_wt_comprador_opp())
             .withColumn('_completa', isin_('status', CRITERIOS_SHOW).cast('int'))
             .withColumn('_fecha_agendada', F.coalesce(fmt_d(ts('sched_start_time')), F.lit(''))))   # NaT -> ''

sum_citas = (ci_comp.groupBy('opportunity_id')
             .agg(F.countDistinct('created_date').alias('_n'),
                  F.min('_fecha_agendada').alias('fecha_primera_cita_visita_comp'),
                  F.max('_fecha_agendada').alias('fecha_ultima_cita_visita_comp'),
                  F.sum('_completa').alias('_comp'))
             .select('opportunity_id',
                     (F.col('_n') > 0).cast('int').alias('flag_cita_agendada_oportunidad'),
                     'fecha_primera_cita_visita_comp', 'fecha_ultima_cita_visita_comp',
                     (F.col('_comp') > 0).cast('int').alias('flag_cita_show_oportunidad')))

# citas "proactivas": sin oportunidad y sin pedido (a nivel id_am)
_ped_pairs = P.select('commerce_order_id', 'sf_order_id').distinct()
pro = (Ci.join(_ped_pairs, 'commerce_order_id', 'left')
         .filter(F.col('opportunity_id').isNull())
         .filter(_wt_comprador_opp())
         .filter(F.col('sf_order_id').isNull())
         .select('id_am', 'status').distinct())
pro_agend_ids = pro.select(F.col('id_am').alias('_k')).filter(F.col('_k').isNotNull()).distinct().withColumn('_f_ag', F.lit(1))
pro_show_ids  = pro.filter(isin_('status', CRITERIOS_SHOW)).select(F.col('id_am').alias('_k')).filter(F.col('_k').isNotNull()).distinct().withColumn('_f_sh', F.lit(1))
# quirk de pandas: Series.isin(valores con NaN) hace match NaN-NaN -> las opps sin id_am_comprador reciben flag=1 si hay citas proactivas sin id_am
_pro_null_ag = pro.filter(F.col('id_am').isNull()).count() > 0
_pro_null_sh = pro.filter(F.col('id_am').isNull() & isin_('status', CRITERIOS_SHOW)).count() > 0
print(f'citas proactivas con id_am nulo (activa quirk NaN de pandas): agendada={_pro_null_ag}, show={_pro_null_sh}')

# pedidos por oportunidad
sum_ped = (P.groupBy('opportunity_id')
            .agg(F.countDistinct('sf_order_id').alias('numero_pedidos'),
                 F.sum(isin_('sf_order_status', STATUS_PEDIDOS_ABIERTOS).cast('int')).alias('numero_pedidos_abiertos'),
                 F.min('fecha_de_creacion').alias('fecha_primer_pedido'),
                 F.max('fecha_de_creacion').alias('fecha_ultimo_pedido')))

# solicitudes de crédito por oportunidad
S_flags = (S.withColumn('_folio', F.col('folio').cast('double'))
             .withColumn('_flag_folio', F.coalesce(F.col('_folio').isNotNull() & (F.col('_folio') > 0), F.lit(False)).cast('int'))
             .withColumn('_flag_aceptada', (contains_('status_solicitud', 'acept') & (F.col('_flag_folio') == 1)).cast('int'))
             .withColumn('_flag_conting', eq('tipo_conclusion_flujo_credito', 'flujo contingente').cast('int')))
sum_sol = (S_flags.groupBy('opportunity_id')
           .agg(F.countDistinct('simulation_name').alias('n_simulaciones_credito'),
                F.sum('_flag_folio').alias('n_simulaciones_con_folio'),
                F.sum('_flag_aceptada').alias('n_simulaciones_preaceptadas'),
                F.sum('_flag_conting').alias('n_simulaciones_contingencia')))

# último cierre perdido (más reciente por oportunidad)
ho_cierre = (first_by(HO.filter(eq('new_value', 'cerrada (perdida)')), ['opportunity_id'],
                      [F.col('created_date').desc(), F.col('_ord').asc()])
             .select('opportunity_id',
                     F.col('old_value').alias('stage_antes_de_cierre'),
                     F.col('created_by').alias('usuario_que_cerro'),
                     nulo().alias('equipo_que_cerro'),
                     F.col('created_date').alias('fecha_cierre')))

# primer booker de la primera cita de comprador de cada oportunidad
ci_aux = (first_by(ci_comp, ['opportunity_id'], [F.col('created_date').asc(), F.col('_ord').asc()])
          .select('opportunity_id', 'numero_cita'))
pb = (first_by(HCi.join(ci_aux.select('numero_cita').distinct(), 'numero_cita', 'left_semi')
                  .filter(eq('field', 'status') & F.col('old_value').isNull() & eq('new_value', 'scheduled')),
               ['numero_cita'], [F.col('created_date').asc(), F.col('_ord').asc()])
      .select('numero_cita',
              F.col('created_by').alias('primer_booker_comprador'),
              nulo().alias('equipo_primer_booker_comprador')))
primer_booker = ci_aux.join(pb, 'numero_cita', 'left').drop('numero_cita')
print('listo')

### 6.3 Join principal (con el mismo fan-out que pandas) y dedupe por oportunidad

In [ ]:
R = (O
     .withColumn('opportunity_owner_equipo', nulo())
     .join(C_sc.select('opportunity_id', 'case_id_perfilamiento_sc', 'case_status_perfilamiento_sc',
                       'case_owner_name_perf_sc', 'case_owner_id_perf_sc', '_ord_sc'), 'opportunity_id', 'left')
     .join(fecha_asig_sc, 'case_id_perfilamiento_sc', 'left')
     .withColumn('case_owner_equipo_perf_sc', nulo())
     .join(C_cr.select('opportunity_id', 'case_id_perfilamiento_credito', 'case_status_perfilamiento_credito', '_ord_cr'),
           'opportunity_id', 'left')
     .join(asesor_sc, 'case_id_perfilamiento_sc', 'left')
     .withColumn('equipo_asesor_caso_tomado_perf_sc', nulo())
     .join(asesor_cr, 'case_id_perfilamiento_credito', 'left')
     .withColumn('equipo_asesor_perfilamiento_credito', nulo())
     .join(sum_ped,   'opportunity_id', 'left')
     .join(sum_sol,   'opportunity_id', 'left')
     .join(sum_citas, 'opportunity_id', 'left'))

# flags cita sin pedido (a nivel id_am_comprador)
R = (R.join(F.broadcast(pro_agend_ids), F.col('id_am_comprador') == F.col('_k'), 'left').withColumnRenamed('_k', '_k_ag')
      .join(F.broadcast(pro_show_ids),  F.col('id_am_comprador') == F.col('_k'), 'left').withColumnRenamed('_k', '_k_sh')
      .withColumn('flag_cita_sin_pedido_agendada',
                  F.when(F.col('_f_ag').isNotNull(), 1)
                   .when(F.col('id_am_comprador').isNull() & F.lit(_pro_null_ag), 1).otherwise(0))
      .withColumn('flag_cita_sin_pedido_show',
                  F.when(F.col('_f_sh').isNotNull(), 1)
                   .when(F.col('id_am_comprador').isNull() & F.lit(_pro_null_sh), 1).otherwise(0))
      .drop('_k_ag', '_k_sh', '_f_ag', '_f_sh')
      .withColumn('flag_caso_tomado_perf_sc',      F.col('fecha_caso_tomado_sc').isNotNull().cast('int'))
      .withColumn('flag_perfilamento_credito',     F.col('fecha_asignacion_perfilamiento_credito').isNotNull().cast('int'))
      .withColumn('opportunity_created_date_day',  fmt_d('opportunity_created_date'))
      .join(ho_cierre,      'opportunity_id', 'left')
      .join(primer_booker,  'opportunity_id', 'left'))

# dedupe: 1 fila por oportunidad. pandas: sort por fecha_asignacion_perfilamiento_credito desc (nulos al final) + keep first;
# en empates conserva el orden de la unión (caso SC, evento, caso crédito) -> aquí se replica con _ord_*.
R = first_by(R, ['opportunity_id'],
             [F.col('fecha_asignacion_perfilamiento_credito').desc_nulls_last(),
              F.col('_ord_sc').asc_nulls_last(), F.col('_ord_ev').asc_nulls_last(), F.col('_ord_cr').asc_nulls_last()])
R = R.drop('_ord_sc', '_ord_ev', '_ord_cr', '_ord')
print('listo')

### 6.4 Origen de crédito calculado (1 simulación vs varias) + PUC de apartados sin simulación

In [ ]:
FECHA_SAFETY = '2100-12-01 11:00:00'

ops = R.filter(F.col('opportunity_source').isin('credito am', 'apartado am')).select('opportunity_id')

sim_n   = S.groupBy('opportunity_id').agg(F.countDistinct('simulation_name').alias('n_simulaciones'))
sim_ult = (first_by(S, ['opportunity_id'], [F.col('created_date').desc(), F.col('_ord').asc()])
           .select('opportunity_id', 'simulation_name', 'tipo_conclusion_flujo_credito', 'status_solicitud',
                   'created_date', 'tipo_credito'))
ops = ops.join(sim_n, 'opportunity_id', 'left').join(sim_ult, 'opportunity_id', 'left')

def _calc_origen(df):
    aux = F.when(eq('tipo_conclusion_flujo_credito', 'flujo contingente'), F.lit('credito am contingencia')).otherwise(F.lit('credito am api'))
    df = df.withColumn('_aux', aux)
    rech = contains_('status_solicitud', 'rechaz')
    suf = (F.when((F.col('_aux') == 'credito am api') & rech, F.lit(' rechazado'))
            .when(F.col('_aux') == 'credito am api', F.lit(' aprobado'))
            .otherwise(F.lit('')))
    return df.select('opportunity_id',
                     F.concat(F.col('_aux'), suf).alias('opportunity_source_calculado'),
                     F.col('simulation_name').alias('simulation_name_op'),
                     'tipo_credito',
                     F.col('created_date').alias('created_date_sim_op'))

# a) exactamente 1 simulación
origen_1 = _calc_origen(ops.filter(F.col('n_simulaciones') == 1))

# b) varias simulaciones: se descartan las posteriores al caso tomado (salvo que todas lo sean) y se elige por tier
multi_ids = ops.filter(F.col('n_simulaciones') > 1).select('opportunity_id')
_w_opp = Window.partitionBy('opportunity_id')
det = (S.join(multi_ids, 'opportunity_id', 'left_semi')
        .join(R.select('opportunity_id', 'fecha_caso_tomado_sc'), 'opportunity_id', 'left')
        .withColumn('_flag_raro',
                    (F.col('created_date') > F.coalesce(ts('fecha_caso_tomado_sc'), F.lit(FECHA_SAFETY).cast('timestamp'))).cast('int'))
        .withColumn('_n_sim',  F.size(F.collect_set('simulation_name').over(_w_opp)))   # nunique
        .withColumn('_n_raro', F.sum('_flag_raro').over(_w_opp))
        .filter((F.col('_flag_raro') != 1) | (F.col('_n_sim') == F.col('_n_raro')))
        .withColumn('_tier',
                    F.when(eq('status_solicitud', 'pre aceptada'), 1)
                     .when(eq('tipo_conclusion_flujo_credito', 'flujo contingente'), 3)
                     .when(eq('status_solicitud', 'rechazada automatica'), 2)))
origen_n = _calc_origen(first_by(det, ['opportunity_id'],
                                 [F.col('_tier').asc_nulls_last(), F.col('created_date').desc(), F.col('_ord').asc()]))

origen = origen_1.unionByName(origen_n)
print(f'opps con origen calculado: {origen.count():,}')

R = (R.join(origen, 'opportunity_id', 'left')
      .withColumn('opportunity_source_aux', F.col('opportunity_source_calculado'))
      .drop('opportunity_source_calculado')
      .filter(F.col('opportunity_created_date_day') >= F.lit(FECHA_CORTE))
      .filter(~isin_('id_am_comprador', ID_AM_DUMMIES)))

# PUC: apartados sin simulación -> se clasifican por score
_score = F.col('perf_bc_score').cast('double')
_puc = (eq('opportunity_source', 'apartado am') & eq('perf_contactado', 'si') & eq('perf_interesado', 'si')
        & isin_('perf_intencion_pago', ['Crédito', 'Credito', 'credito'])
        & F.col('perf_bc_score').isNotNull() & F.col('opportunity_source_aux').isNull())
R = R.withColumn('opportunity_source_aux',
                 F.when(_puc & (_score <= 569), 'apartado puc no apto')
                  .when(_puc & _score.between(570, 649), 'apartado puc viable kuna')
                  .when(_puc & _score.between(650, 699), 'apartado puc viable bbva')
                  .when(_puc & (_score >= 700), 'apartado puc directo eam')
                  .otherwise(F.col('opportunity_source_aux')))

_faltan = [c for c in COLS_OPORTUNIDADES if c not in R.columns]
rep_opp = R.select([c for c in COLS_OPORTUNIDADES if c in R.columns]).cache()
_n_opp = rep_opp.count()
print(f'reporte_oportunidades: {_n_opp:,} filas, {len(rep_opp.columns)} cols')
if _faltan:
    print(f'  [INFO] columnas de la config que no existen: {_faltan}')
assert _n_opp == rep_opp.select('opportunity_id').distinct().count(), 'oportunidades: opportunity_id duplicado'

## 7. `reporte_citas`
(usa `rep_opp` = `oportunidades_consumo`, como en pandas)

In [ ]:
# oportunidades reducidas (fechas a yyyy-MM-dd como en pandas)
oppss = (rep_opp.select('opportunity_id', 'opportunity_name', F.col('owner_id').alias('opportunity_owner_id'),
                        'opportunity_owner', 'opportunity_owner_equipo', 'perf_bc_score', 'perf_intencion_pago',
                        'opportunity_stage',
                        fmt_d('opportunity_created_date').alias('opportunity_created_date'),
                        fmt_d(F.to_date('fecha_asignacion', 'dd/MM/yyyy')).alias('fecha_asignacion'),
                        'id_am_comprador', 'opportunity_source', 'opportunity_source_aux'))

clientes = CL.select(F.col('id_am').cast('long').alias('id_am'),
                     F.col('nickname').alias('nombre'), 'email', F.col('phone').alias('telefono'))
ped_c = P.select('commerce_order_id', 'sf_order_id')

hci = (HCi.withColumnRenamed('created_by_id', 'booker_id').withColumnRenamed('created_by', 'booker_name'))
_h = lambda: hci.select('numero_cita', 'booker_id', 'booker_name', 'field', 'old_value', 'new_value', 'created_date', '_ord')

CC = (Ci
      .withColumnRenamed('created_date', 'created_time')
      .withColumnRenamed('territorio_cita', 'espacio_cita')
      .withColumnRenamed('fecha_checkin', 'check_in')
      .withColumnRenamed('fecha_checkout', 'check_out')
      .withColumn('_wtl', F.lower('work_type_name'))
      .withColumn('rol', F.when(isin_('_wtl', WT_COMPRADOR), 'comprador').when(isin_('_wtl', WT_VENDEDOR), 'vendedor').otherwise(''))
      .drop('_wtl')
      .withColumn('sched_date',   fmt_d(ts('sched_start_time')))
      .withColumn('created_date', fmt_d('created_time'))
      .withColumn('id_am', F.col('id_am').cast('long'))
      .join(clientes, 'id_am', 'left')
      .join(ped_c, 'commerce_order_id', 'left')
      .join(oppss.select('opportunity_id', 'opportunity_owner_id', 'opportunity_owner', 'perf_bc_score', 'perf_intencion_pago',
                         'opportunity_stage', 'opportunity_created_date', 'fecha_asignacion', 'id_am_comprador',
                         'opportunity_source', 'opportunity_source_aux'), 'opportunity_id', 'left'))

# booker original (primer evento status null -> scheduled)
booker = (first_by(_h().filter(eq('field', 'status') & F.col('old_value').isNull() & eq('new_value', 'scheduled')),
                   ['numero_cita'], [F.col('created_date').asc(), F.col('_ord').asc()])
          .select('numero_cita', 'booker_id', 'booker_name'))
# respaldo: quién creó la cita (primer evento 'created' con usuario, en orden físico)
creador = (first_by(_h().filter(eq('field', 'created') & F.col('booker_name').isNotNull()),
                    ['numero_cita'], [F.col('_ord').asc()])
           .select('numero_cita', F.col('booker_id').alias('hcita_created_by_id'), F.col('booker_name').alias('hcita_created_by')))

CC = (CC.join(booker, 'numero_cita', 'left')
        .join(creador, 'numero_cita', 'left')
        .withColumn('flag_booker_origen_nulo', F.col('booker_id').isNull().cast('int'))
        .withColumn('booker_id', F.coalesce('booker_id', 'hcita_created_by_id'))
        .withColumn('booker_name', F.upper(F.trim(unaccent(F.coalesce('booker_name', 'hcita_created_by')))))   # normalize + upper
        .drop('hcita_created_by_id', 'hcita_created_by'))
CC = add_team(CC, 'opportunity_created_date', 'opportunity_owner_id', 'opportunity_owner_equipo')
CC = add_team(CC, 'created_date', 'booker_id', 'booker_equipo')

# flags y valores por defecto
CC = (CC
      .withColumn('sf_order_id',     F.coalesce('sf_order_id', F.lit(-1)))
      .withColumn('rol',             F.when(F.col('rol') == '', 'desconocido').otherwise(F.col('rol')))
      .withColumn('work_type_name',  F.when(F.col('work_type_name').isNull() | (F.col('work_type_name') == ''), 'desconocido').otherwise(F.col('work_type_name')))
      .withColumn('booker_name',     F.coalesce('booker_name', F.lit('desconocido')))
      .withColumn('booker_equipo',   F.coalesce('booker_equipo', F.lit('desconocido')))
      .withColumn('id_am',           F.coalesce('id_am', F.lit(-1).cast('long')))
      .withColumn('flag_cita_show_comprador', isin_('status', CRITERIOS_SHOW).cast('int'))
      .withColumn('_agend_tmp', (isin_('rol', CRITERIOS_AGEND['rol']) & isin_('work_type_name', CRITERIOS_AGEND['wtn'])).cast('int'))
      .withColumn('flag_dummy', ((F.col('opportunity_id').isNotNull() & (F.col('flag_booker_origen_nulo') == 1))
                                 | (isin_('id_am', ID_AM_DUMMIES) & (F.col('_agend_tmp') == 1))).cast('int'))
      .withColumn('flag_cita_agendada_comprador', ((F.col('_agend_tmp') == 1) & (F.col('flag_dummy') == 0)).cast('int')))

_w_dup = (Window.partitionBy('id_am', 'work_type_name', 'rol', 'sched_date', 'sf_order_id')
          .orderBy(F.col('rol').asc(), F.col('flag_dummy').asc(), F.col('sf_order_id').desc(),
                   F.col('flag_cita_show_comprador').desc(), F.col('created_date').desc(), F.col('_ord').asc()))
CC = (CC.withColumn('flag_duplicada', (F.row_number().over(_w_dup) > 1).cast('int'))
        .drop('_agend_tmp'))

# dueño del caso de perfilamiento contact center (pandas lo une SIN deduplicar -> el assert del final lo vigila)
casos_sc = (C.filter(F.col('opportunity_id').isNotNull() & eq('case_subject', 'perfilamiento contact center'))
            .select('opportunity_id',
                    F.col('case_owner_name_perf_sc').alias('perf_sc_case_owner_name'),
                    F.col('case_owner_id_perf_sc').alias('perf_sc_case_owner_id')))
CC = CC.join(casos_sc, 'opportunity_id', 'left')

# quién canceló la cita (último evento de status con valor de cancelación / no show)
cancel = (first_by(_h().filter(eq('field', 'status') & isin_('new_value', CRITERIOS_CANCELADA)),
                   ['numero_cita'], [F.col('created_date').desc(), F.col('_ord').asc()])
          .select('numero_cita',
                  F.col('new_value').alias('status_cancelacion'),
                  F.col('booker_id').alias('cancelado_por_id'),
                  F.col('booker_name').alias('cancelado_por')))
CC = CC.join(cancel, 'numero_cita', 'left')
CC = add_team(CC, 'created_date', 'cancelado_por_id', 'cancelado_por_equipo')
_aplica_cancel = F.coalesce(F.col('status').rlike('cancelado|no show'), F.lit(False))   # str.contains(regex, na=False)
for _c_ in ['status_cancelacion', 'cancelado_por_id', 'cancelado_por', 'cancelado_por_equipo']:
    CC = CC.withColumn(_c_, F.when(_aplica_cancel, F.col(_c_)))   # fuera de cancelado/no show -> null

# relleno de perfilamiento: citas sin opportunity_id (desde 2026-08-17) toman la oportunidad más cercana anterior del mismo id_am
_rell = [c for c in COLS_RELLENAR]
oppss_rell = (oppss.select(F.col('id_am_comprador').cast('long').alias('id_am'), *[F.col(c).alias(c + '_relleno') for c in _rell])
                   .filter(F.col('id_am').isNotNull() & (F.col('id_am') != -1)))
tgt = (CC.filter(F.col('created_date') >= F.lit(FECHA_RELLENO_CITAS))
         .filter(F.col('opportunity_id').isNull())
         .filter(F.col('id_am').isNotNull() & (F.col('id_am') != -1))
         .filter(isin_('rol', ['comprador', 'desconocido']))
         .select('numero_cita', 'id_am', 'created_date')
         .join(oppss_rell, 'id_am', 'left')
         .withColumn('_diff', F.datediff(F.to_date('created_date'), F.to_date('opportunity_created_date_relleno')))
         .filter(F.col('opportunity_created_date_relleno').isNotNull() & (F.col('_diff') >= 0)))
tgt = (first_by(tgt, ['numero_cita'], [F.col('_diff').asc(), F.col('opportunity_id_relleno').asc()])
       .select('numero_cita', *[c + '_relleno' for c in _rell]))

CC = (CC.join(tgt, 'numero_cita', 'left')
        .withColumn('flag_oppid_origen_nulo', F.col('opportunity_id').isNull().cast('int')))
for _c_ in _rell:
    CC = CC.withColumn(_c_, F.coalesce(F.col(_c_), F.col(_c_ + '_relleno')))
CC = CC.drop(*[c + '_relleno' for c in _rell])

# agrupación operativa
_op_map = F.create_map(*[x for kv in CRM_EQUIPOS_OPERATIVOS.items() for x in (F.lit(kv[0]), F.lit(kv[1]))])
CC = (CC.withColumn('booker_equipo_operativo',         F.coalesce(_op_map[F.col('booker_equipo')], F.lit('desconocido')))
        .withColumn('cancelado_por_equipo_operativo',  F.coalesce(_op_map[F.col('cancelado_por_equipo')], F.lit('desconocido'))))

_faltan = [c for c in COLS_CITAS if c not in CC.columns]
rep_citas = CC.select([c for c in COLS_CITAS if c in CC.columns]).orderBy(F.col('numero_cita').desc()).cache()
_n_citas = rep_citas.count()
print(f'reporte_citas: {_n_citas:,} filas, {len(rep_citas.columns)} cols')
if _faltan:
    print(f'  [INFO] columnas de la config que no existen: {_faltan}')

# mismos asserts que pandas (detectan fan-out por llaves repetidas en pedidos / casos)
assert _n_citas == Ci.count(), f'citas: {_n_citas} filas vs {Ci.count()} de entrada -> algún join duplicó filas'
assert rep_citas.select('numero_cita').distinct().count() == Ci.select('numero_cita').distinct().count()

## 8. Validación de esquema antes de escribir

In [ ]:
def resumen(nombre, df, esperadas, clave=None):
    print(f'\n=== {nombre}: {df.count():,} filas | {len(df.columns)} cols')
    print('   columnas fuera de orden o faltantes vs config:',
          [c for c in esperadas if c not in df.columns] or 'ninguna')
    if clave:
        print(f'   clave {clave}: nulos={df.filter(F.col(clave).isNull()).count()}, '
              f'duplicados={df.count() - df.select(clave).distinct().count()}')
    print('   ' + ', '.join(f'{n}:{t}' for n, t in df.dtypes))

resumen('reporte_oportunidades', rep_opp, COLS_OPORTUNIDADES, 'opportunity_id')
resumen('reporte_citas', rep_citas, COLS_CITAS, 'numero_cita')
resumen('reporte_simulaciones', rep_sim, COLS_SIMULACIONES)
resumen('reporte_historico_oportunidades', rep_hist, ['opportunity_id', 'old_value', 'new_value', 'created_by', 'created_date', 'opportunity_name'])

## 9. Escritura en Gold S3

In [ ]:
SALIDAS = {
    'ReporteOportunidades':          rep_opp,
    'ReporteHistoricoOportunidades': rep_hist,
    'ReporteCitas':                  rep_citas,
    'ReporteSimulaciones':           rep_sim,
}
for nombre, df in SALIDAS.items():
    path = GOLD + nombre + '/'
    if ESCRIBIR:
        df.coalesce(1).write.mode('overwrite').parquet(path)   # coalesce(1) conserva el orden del orderBy
        print(f'Escrito {nombre} -> {path}')
    else:
        print(f'[ESCRIBIR=False] omitido {nombre} -> {path}')
print(f'Fin: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}')